# LOOP-01 · Integer-N CP-PLL 逐边沿实验

40 MHz × 125 = 5 GHz，起始频偏 −80 MHz。按顺序验证 PFD/Divider、CP 脉冲、VCO SPICE 扫描、闭环与平均模型。原始数据、网表、日志和网页回放一并导出。

需 Python 3.12、Node.js 24、npm 和 unzip。首次运行下载固定仓库版本并安装锁定的 ngspice WASM。

In [ ]:
ICP_UA = 100
R_OHM = 4700

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv
import io
import json
import shutil
import subprocess
import tarfile
import tempfile

source_commit = '353804889ec9671b90e9d37627bc5e92259417cf'
archive_url = f'https://codeload.github.com/thecloud723/analog-ic-learning/tar.gz/{source_commit}'
workspace = Path(tempfile.mkdtemp(prefix='loop01-notebook-', dir=Path.cwd()))
with urlopen(archive_url, timeout=60) as download:
    with tarfile.open(fileobj=io.BytesIO(download.read()), mode='r:gz') as archive:
        archive.extractall(workspace, filter='data')
repo = workspace / f'analog-ic-learning-{source_commit}'
lab = repo / 'pll/labs/PLL-SPICE'
baseline = workspace / 'baseline'
print('Source:', source_commit)
print('Node:', subprocess.check_output(['node', '--version'], text=True).strip())

## 标准实验

Run All 生成 PFD 断言、ngspice 脉冲、PLL-01 调谐、闭环扫描、PLL-03 对照和网页回放。

In [ ]:
with (workspace / 'reproduce.log').open('w') as log:
    subprocess.run(['bash', lab / 'reproduce_loop01.sh', baseline], check=True,
                   stdout=log, stderr=subprocess.STDOUT, cwd=repo)
pulse = json.loads((baseline / 'pulse/comparison.json').read_text())
cases = json.loads((baseline / 'nonideal/comparison.json').read_text())
pfd = (baseline / 'pfd.jsonl').read_text().splitlines()
average = json.loads((baseline / 'average/comparison.json').read_text())
assert len(pfd) == 20 and len(pulse) == 12 and len(cases) == 12
assert all(case['run']['lock'] for case in cases)
print(f'{len(pfd)} PFD cases · {len(pulse)} SPICE pulse cases · {len(cases)} capture cases')
print(f"Peak SPICE pulse discrepancy: {max(row['maxNodeError'] for row in pulse)*1e9:.3f} nV")
print(f"Event vs averaged entry difference: {average['entryDifferenceUs']:.3f} µs")

## 改参重跑

修改顶部参数格的电荷泵电流和串联电阻，运行标准实验及下方的 5% 失配、5 pF 并联电容工况，保留独立的 CSV 和结果 ZIP。

In [ ]:
common = json.loads((lab / 'settings.json').read_text())
common.update(cp_uA=ICP_UA, loop_r_ohm=R_OHM)
model = json.loads((baseline / 'calibrated.json').read_text())
model['useShuntCap'] = True
common_path, model_path = workspace / 'common.json', workspace / 'model.json'
common_path.write_text(json.dumps(common, indent=2))
model_path.write_text(json.dumps(model, indent=2))
summary = json.loads(subprocess.check_output(
    ['node', lab / 'run_loop01.mjs', workspace / 'parameter-run', common_path, model_path], text=True))
assert summary['lock']
print(f"ICP={ICP_UA:g} µA, R={R_OHM:g} Ω")
print(f"Entry {summary['lock']['entryUs']:.3f} µs, confirmation {summary['lock']['confirmationUs']:.3f} µs")
print(f"Final control voltage: {summary['finalVctrlV']:.6f} V")

## 波形和结果导出

频差为每参考周期的平均值。5 pF 并联电容使 VCTRL 与串联支路电容成为两个独立状态。filter_events.csv 保留上下拉逻辑、实际泵开关及事件电流。

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

run = Path(summary['directory'])
def rows(path):
    with path.open() as f:
        return list(csv.DictReader(f))

frequency = rows(run / 'samples.csv')
voltages = rows(run / 'filter_samples.csv')
t = [float(row['time_ns']) / 1000 for row in frequency]
fig, (ax, voltage) = plt.subplots(2, 1, figsize=(7.0, 4.0), sharex=True, layout='constrained')
ax.plot(t, [float(row['frequency_error_hz']) / 1e6 for row in frequency])
ax.axvline(summary['lock']['entryUs'], color='gray', linestyle='--')
ax.set_ylabel('Frequency error (MHz)')
voltage.plot(t, [float(row['vctrl_v']) for row in voltages], label='VCTRL')
voltage.plot(t, [float(row['capacitor_v']) for row in voltages], '--', label='Series capacitor')
voltage.set(xlabel='Time (µs)', ylabel='Voltage (V)')
voltage.legend(loc='lower right')
fig.savefig(workspace / 'lock-waveform.png', dpi=85)
plt.close(fig)
saved = {**summary, 'notebookSourceCommit': source_commit, 'ICP_UA': ICP_UA, 'R_OHM': R_OHM,
         'pfdCases': len(pfd), 'pulseCases': len(pulse), 'captureCases': len(cases)}
(workspace / 'notebook-summary.json').write_text(json.dumps(saved, indent=2))
shutil.rmtree(repo)  # The pinned archive remains available through archive_url.
bundle = shutil.make_archive(str(workspace) + '-results', 'zip', root_dir=workspace)
print('Saved waveform, raw CSV, netlists, settings, logs, and replay.')
print('Results ZIP:', Path(bundle).name)